In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from google import genai

# 1. Initialize the client
client = genai.Client()

# 2. Call the models service
# response = client.models.generate_content(
#     model="gemini-2.5-pro",  # Or gemini-2.5-pro depending on your preference
#     contents="Your prompt here"
# )
# print(response.text)

In [17]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",  # "gemini-2.5-flash" is great for speed/cost, or use "gemini-2.5-pro" for complex tasks
        contents=prompt
    )
    return response.text

In [18]:
llm('hey whats up?')

"Hey there! Not much, just here and ready to chat.\n\nWhat's up with you? Anything I can help with, or just saying hi?"

In [19]:
question = 'I just discorver the course. Can I join now'
answer = llm(question)
print(answer)

That's great you've discovered a course you're interested in!

To give you the most accurate answer, I'll need a little more information about **which course you're referring to.**

Could you please tell me:

1.  **What's the name of the course?**
2.  **Who is offering it?** (e.g., a university, an online platform like Coursera/edX/Udemy, a private training provider, etc.)
3.  **Is it a self-paced course or one with a fixed start date?**

**In general, here's how it often works:**

*   **For many self-paced online courses (like those on platforms such as Coursera, Udemy, or LinkedIn Learning):** Yes, you can usually enroll and start immediately at any time.
*   **For cohort-based courses, university courses, or programs with specific start dates:** There are often enrollment periods and deadlines. If you've just discovered it, you might be able to join the current cohort, or you might have to wait for the next one.

Once I have more details, I can help you find the specific enrollment 

In [29]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [30]:
prompt = f"""   
Your task is to answer questions from the course participants based on the provided context.    

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,   
respond with "I dont know." 

Question:   
{question}      

Context:    
{context}
"""

In [31]:
answer = llm(prompt)    
print(answer)

Yes, you can join now, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
# RAG code process that we will use throughout the workshop
def rag(question):  
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 83}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [36]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [38]:
documents[1345]

{'id': 'ab183bd688',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it."}